# Scihe notebook feature test

Run cells with `SPC n r` (run), `SPC n n` (run and advance) or `SPC n a` (run all).
Outputs appear inline below each cell; `SPC n v` shows a cell's full output.

The terminal shows text only: images appear as a placeholder line
(`SPC n I` / `:nbimage` opens them with the system viewer), and LaTeX,
HTML and markdown math are shown as their source. All of them are kept
in the `.ipynb` file, so Jupyter shows them rendered.

Sections: plain output and state · stdout/stderr · long output · rich display ·
LaTeX (shown as source) · images (placeholders) · errors · interrupt · shell commands.

## 1. Expressions, print and kernel state

In [ ]:
x = 6 * 7
print("printed:", x)
x  # the last expression is shown as the cell result

In [ ]:
# state persists between cells
y = x + 1
f"y = {y}", type(y).__name__

## 2. stdout and stderr

In [ ]:
import sys
print("to stdout")
print("to stderr", file=sys.stderr)

## 3. Long output (folded inline; `SPC n x` toggles folding, `SPC n v` shows all)

In [ ]:
for i in range(100):
    print(f"line {i:3d}", "#" * (i % 40))

## 4. Rich display: several outputs from one cell

In [ ]:
display("first display call")
display({"a": 1, "b": [1, 2, 3]})

class Html:
    def _repr_html_(self):
        return "<b>bold</b> HTML (kept in the file, shown as text here)"
    def __repr__(self):
        return "Html()"

display(Html())
"done"

## 5. LaTeX (kept in the file, shown as source)

Markdown cells can contain math, e.g. inline $e^{i\pi} + 1 = 0$ and a display formula:

$$\int_{-\infty}^{\infty} e^{-x^2}\,dx = \sqrt{\pi}$$

Scihe does not render it: the formulas stay as written here.
The cells below produce `text/latex` outputs from code; they show their
text form inline, and Jupyter renders them.

In [ ]:
class Latex:
    # minimal object with a LaTeX representation (no IPython needed)
    def __init__(self, src):
        self.src = src
    def _repr_latex_(self):
        return self.src
    def __repr__(self):
        return self.src

display(Latex(r"$$\sum_{k=1}^{n} k = \frac{n(n+1)}{2}$$"))
display(Latex(r"$$\nabla \cdot \mathbf{E} = \frac{\rho}{\varepsilon_0}$$"))
Latex(r"$\hat{\beta} = (X^\top X)^{-1} X^\top y$")

In [ ]:
# the same through IPython, when it is installed
try:
    from IPython.display import Math, Latex as IPyLatex
    display(Math(r"f(x) = \frac{1}{\sigma\sqrt{2\pi}}\, e^{-\frac{(x-\mu)^2}{2\sigma^2}}"))
    display(IPyLatex(r"Euler: $e^{i\theta} = \cos\theta + i\sin\theta$"))
except ImportError:
    print("IPython not installed: skipped")

## 6. Images with matplotlib (shown as a placeholder)

Figures are captured automatically and saved in the notebook. Inline you
see a line like `[image/png 24.1 KB — :nbimage to view]`; `SPC n I`
opens the images with the system viewer.

In [ ]:
import math
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib not installed: skipped")

if plt:
    xs = [i / 20 for i in range(200)]
    plt.figure(figsize=(6, 3))
    plt.plot(xs, [math.sin(v) for v in xs], label=r"$\sin(x)$")
    plt.plot(xs, [math.cos(v) for v in xs], label=r"$\cos(x)$")
    plt.title(r"Trigonometric functions: $\sin^2 x + \cos^2 x = 1$")
    plt.legend()
    plt.grid(alpha=0.3)

In [ ]:
# two figures from one cell -> two image outputs
if plt:
    import random
    random.seed(1)
    data = [random.gauss(0, 1) for _ in range(2000)]
    fig1, ax1 = plt.subplots(figsize=(4, 3))
    ax1.hist(data, bins=30, color="tab:blue")
    ax1.set_title(r"$\mathcal{N}(0, 1)$ sample")
    fig2, ax2 = plt.subplots(figsize=(4, 3))
    ax2.scatter(data[:-1:4], data[1::4], s=4, alpha=0.5)
    ax2.set_title("lag plot")

## 7. An image without matplotlib (`_repr_png_`, shown as a placeholder)

In [ ]:
import struct, zlib

class Png:
    # a tiny gradient PNG built by hand
    def __init__(self, w=64, h=64):
        rows = b"".join(
            b"\x00" + b"".join(bytes((x * 255 // (w - 1), y * 255 // (h - 1), 160)) for x in range(w))
            for y in range(h))
        def chunk(t, d):
            return struct.pack(">I", len(d)) + t + d + struct.pack(">I", zlib.crc32(t + d) & 0xFFFFFFFF)
        self.data = (b"\x89PNG\r\n\x1a\n" + chunk(b"IHDR", struct.pack(">IIBBBBB", w, h, 8, 2, 0, 0, 0))
                     + chunk(b"IDAT", zlib.compress(rows)) + chunk(b"IEND", b""))
    def _repr_png_(self):
        return self.data
    def __repr__(self):
        return "<gradient PNG>"

Png()

## 8. Errors and tracebacks

Like Jupyter, "run all" stops at a failing cell. Continue with `SPC n B`
(run this cell and below) from the next cell.

In [ ]:
def broken():
    return 1 / 0

broken()

## 9. Interrupt: run this, then press `SPC n i` (the kernel keeps its state)

In [ ]:
import time
for i in range(30):
    print("tick", i)
    time.sleep(1)

In [ ]:
# still alive after the interrupt
x, y

## 10. Shell commands

A line starting with `!` runs in the shell, like in Jupyter; its output
(stderr included) appears in the cell. `SPC n i` interrupts it.

In [ ]:
!echo "hello from $SHELL"
!python3 -c "import sys; print('child python', sys.version.split()[0], sys.argv[1:])" a b
for name in ["x", "y"]:
    !echo loop
print("back in Python")